# colab_07 — Stratified subsampling to 100k per dataset

## Motivation

Session 15 showed DPT fails on the integrated Harmony object. Root causes: (1) ca. 100× cell imbalance between Bhaduri 2020 organoids (ca. 242k) and Zhong 2018 fetal (ca. 2.4k); (2) disconnected graph components; (3) stale `iroot` propagation through `.copy()`.

Session 16 pivoted: replace Zhong with **Bhaduri 2021** (atlas of cortical arealization; ca. 396k fetal cells). Same lab, same 10x v2 chemistry as Bhaduri 2020 organoids.

Session 17 downloaded Bhaduri 2021 and saved `bhaduri_2021_raw.h5ad` (396,186 × 33,694) on Drive.

**This notebook builds a balanced 1:1 input for integration:** stratified subsample both datasets to 100k cells each. Stratum floor = 200 cells (hard-drop, no upsampling — upsampling duplicates cells at zero distance in the kNN graph, which is exactly the pathology that broke DPT). Per-stratum targets = proportional to cleaned stratum size (preserves natural composition).

Stratification axes:
- **Bhaduri 2020** (organoids): `protocol × age_week` (organoids have no anatomical area).
- **Bhaduri 2021** (fetal): `age_gw × cell_type_coarse`. `area_ucsc` deliberately left out — it would only apply to the fetal side (asymmetric), fragment strata from ca. 120 → ca. 1,200 (most below the 200 floor), and areal identity is a Phase-2 question.

## Section 0 — Setup

Install `scanpy` (fresh Colab kernel) and mount Drive for persistent h5ad storage.

### 0a — Install scanpy and mount Drive

In [1]:
!pip install -q scanpy
from google.colab import drive
drive.mount('/content/drive')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.3/174.3 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 104.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 116.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
Mounted at /content/drive


## Section 1 — Load Bhaduri 2020 and inspect metadata

Starting file is `bhaduri_2020_clustered.h5ad` — contains raw counts in `adata.raw`, Leiden labels, and sample prefixes on barcodes. Integration-stage transforms (HVG selection, PCA, Harmony) will be redone downstream on the balanced input, so we start from the clustered h5ad rather than the dense-scaled preprocessed one (which was ca. 32 GB anyway and has been deleted).

Key question for subsampling: **does the `obs` already contain `protocol` and `age` columns?** If not, we must derive them from the `sample` prefix.

### 1a — Load Bhaduri 2020 and inspect metadata

In [2]:
import scanpy as sc

adata = sc.read_h5ad('/content/drive/MyDrive/brain-organoid-trajectories/data/processed/bhaduri_2020/bhaduri_2020_clustered.h5ad')
print("Shape:", adata.shape)
print("\nobs columns:", list(adata.obs.columns))
print("\nFirst 5 obs rows:")
print(adata.obs.head())
print("\nUnique samples (first 10):", adata.obs['sample'].unique()[:10] if 'sample' in adata.obs.columns else "NO 'sample' COLUMN")
print("\nSample count:", adata.obs['sample'].nunique() if 'sample' in adata.obs.columns else "N/A")

Shape: (241776, 16774)

obs columns: ['n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'sample', 'leiden']

First 5 obs rows:
                           n_genes_by_counts  total_counts  total_counts_mt  \
H1SWeek3_AAACCTGAGACAAAGG               1748  53458.578125       435.720520   
H1SWeek3_AAACCTGAGCACACAG               1253  31004.234375       285.992981   
H1SWeek3_AAACCTGAGGATGGAA               1641  42572.929688       361.858154   
H1SWeek3_AAACCTGCAATTGCTG               1800  65508.453125       139.779099   
H1SWeek3_AAACCTGCAGCGTAAG               1979  70000.132812       722.343750   

                           pct_counts_mt  n_genes    sample leiden  
H1SWeek3_AAACCTGAGACAAAGG       0.815062     1748  H1SWeek3      5  
H1SWeek3_AAACCTGAGCACACAG       0.922432     1253  H1SWeek3      5  
H1SWeek3_AAACCTGAGGATGGAA       0.849972     1641  H1SWeek3      5  
H1SWeek3_AAACCTGCAATTGCTG       0.213376     1800  H1SWeek3      5  
H1SWeek3_AAACCTGCAG

### Finding

- Shape: **241,776 × 16,774**.
- `obs` columns: `n_genes_by_counts`, `total_counts`, `total_counts_mt`, `pct_counts_mt`, `n_genes`, `sample`, `leiden`. **No `protocol`, no `age`** — must be derived.
- 37 unique samples. Names like `H1SWeek3`, `H28126SWeek5`, `YH10PWeek10` — clearly encode **`{cell_line}{protocol_letter}Week{N}`** where protocol ∈ {S, X, P} (matches the paper's 3-protocol design).

## Section 2 — First parsing attempt

Hypothesis: every sample name follows the clean pattern `^{line}{[SXP]}Week{\d+}$`. One regex should cover all 37.

### 2a — First regex attempt

In [3]:
import re
pattern = re.compile(r'^(?P<line>.+?)(?P<protocol>[SXP])Week(?P<age>\d+)$')

all_samples = adata.obs['sample'].unique().tolist()
parsed, failed = [], []
for s in all_samples:
    m = pattern.match(s)
    if m:
        parsed.append((s, m.group('line'), m.group('protocol'), int(m.group('age'))))
    else:
        failed.append(s)

print(f"Parsed: {len(parsed)}/{len(all_samples)}")
print(f"Failed: {failed if failed else 'none'}")

import pandas as pd
df = pd.DataFrame(parsed, columns=['sample', 'line', 'protocol', 'age_week'])
print("\nProtocol counts (samples):", df['protocol'].value_counts().to_dict())
print("Age counts (samples):",       df['age_week'].value_counts().sort_index().to_dict())
print("Line counts (samples):",      df['line'].value_counts().to_dict())
print("\nProtocol × age grid (samples):")
print(df.groupby(['protocol', 'age_week']).size().unstack(fill_value=0))

Parsed: 28/37
Failed: ['H28126S2Week8', 'H28126S2Week5', 'Week5S', 'Week8S', 'Week8P', 'Week5P', 'H28126S2Week10', 'Week10S', 'Week10P']

Protocol counts (samples): {'S': 14, 'X': 8, 'P': 6}
Age counts (samples): {3: 4, 5: 7, 8: 7, 10: 6, 15: 1, 24: 3}
Line counts (samples): {'H28126': 10, 'H1': 9, 'YH10': 6, 'L13234': 3}

Protocol × age grid (samples):
age_week  3   5   8   10  15  24
protocol                        
P          0   2   2   1   0   1
S          2   3   3   3   1   2
X          2   2   2   2   0   0


### Finding

**28/37 parse cleanly. 9 fail.** Two pattern families escape the regex:

1. **`H28126S2Week{5,8,10}`** — cell line `H28126` with `S2` designation (likely protocol S, variant/batch 2). The trailing digit breaks the `{protocol_letter}Week` pattern.
2. **`Week{5,8,10}{S,P}`** — reverse-order naming with no cell-line prefix.

Nine samples sounds small, but we don't know yet how many *cells* they represent — need to check before deciding how careful to be.

## Section 3 — Investigate failed samples

If the 9 unmatched samples carry only a handful of cells, we could drop them. If they carry tens of thousands, we must parse them properly.

### 3a — Count cells in the 9 unmatched samples

In [4]:
failed_samples = ['H28126S2Week8', 'H28126S2Week5', 'Week5S', 'Week8S',
                  'Week8P', 'Week5P', 'H28126S2Week10', 'Week10S', 'Week10P']
print("Cells per unmatched sample:")
print(adata.obs['sample'].value_counts().loc[failed_samples])
print(f"\nTotal unmatched cells: {adata.obs['sample'].isin(failed_samples).sum()}")
print(f"Of grand total: {adata.obs['sample'].isin(failed_samples).sum() / len(adata) * 100:.1f}%")

Cells per unmatched sample:
sample
H28126S2Week8      9198
H28126S2Week5      3784
Week5S            10607
Week8S             5591
Week8P             6825
Week5P             2160
H28126S2Week10     3071
Week10S            6167
Week10P           10315
Name: count, dtype: int64

Total unmatched cells: 57718
Of grand total: 23.9%


### Finding

**57,718 cells (23.9% of the dataset)** are in the 9 unmatched samples. Too large to drop. We must extend the parser.

### Decisions on the two edge-case families

**`H28126S2Week*` (3 samples, 16,053 cells) — fold `S2` into `S`.**
- Our stratification axis is *protocol* for balancing against fetal maturation, not batch-effect analysis.
- Folding matches the paper's stated 3-protocol design.
- Alternative (keep S2 separate) would fragment strata and add a 4th protocol level that isn't biologically distinct.

**`Week{N}{S,P}` (6 samples, 41,665 cells) — label cell line as `unknown`; parse protocol + age normally.**
- Cell line is not a stratification axis, so its absence is acceptable.
- Protocol + age are what we actually need.

## Section 4 — Final parser

Three patterns, tried in order of specificity:

1. `pat_variant` — `{line}{[SXP]}\d+Week{N}` — catches `H28126S2Week*`.
2. `pat_standard` — `{line}{[SXP]}Week{N}` — original clean pattern.
3. `pat_reverse` — `Week{N}{[SXP]}` — catches `Week{N}{S,P}`; line labeled `unknown`.

We then map `(protocol, age_week)` onto every cell and compute the **cell-level grid**, which is what matters for stratification sizing.

### 4a — Final three-pattern parser

In [5]:
import re

pat_standard = re.compile(r'^(?P<line>.+?)(?P<protocol>[SXP])Week(?P<age>\d+)$')
pat_variant  = re.compile(r'^(?P<line>.+?)(?P<protocol>[SXP])\d+Week(?P<age>\d+)$')
pat_reverse  = re.compile(r'^Week(?P<age>\d+)(?P<protocol>[SXP])$')

def parse_sample(s):
    for pat in (pat_variant, pat_standard):  # try variant first (more specific)
        m = pat.match(s)
        if m:
            return m.group('line'), m.group('protocol'), int(m.group('age'))
    m = pat_reverse.match(s)
    if m:
        return 'unknown', m.group('protocol'), int(m.group('age'))
    return None

all_samples = adata.obs['sample'].unique().tolist()
parsed, failed = [], []
for s in all_samples:
    r = parse_sample(s)
    if r:
        parsed.append((s, *r))
    else:
        failed.append(s)

import pandas as pd
df = pd.DataFrame(parsed, columns=['sample', 'line', 'protocol', 'age_week'])

print(f"Parsed: {len(parsed)}/{len(all_samples)}")
print(f"Failed: {failed if failed else 'none'}")
print("\nProtocol × age grid (SAMPLES):")
print(df.groupby(['protocol', 'age_week']).size().unstack(fill_value=0))

# Build two separate mappings (tuple-map on a Categorical series triggers a pandas MultiIndex error)
protocol_map = dict(zip(df['sample'], df['protocol']))
age_map      = dict(zip(df['sample'], df['age_week']))

sample_str = adata.obs['sample'].astype(str)
obs_tmp = pd.DataFrame({
    'protocol': sample_str.map(protocol_map),
    'age_week': sample_str.map(age_map),
})

print("\nProtocol × age grid (CELLS):")
print(obs_tmp.groupby(['protocol', 'age_week']).size().unstack(fill_value=0))
print(f"\nTotal cells with (protocol, age): {obs_tmp.dropna().shape[0]} / {len(obs_tmp)}")

Parsed: 37/37
Failed: none

Protocol × age grid (SAMPLES):
age_week  3   5   8   10  15  24
protocol                        
P          0   3   3   2   0   1
S          2   5   5   5   1   2
X          2   2   2   2   0   0

Protocol × age grid (CELLS):
age_week     3      5      8      10    15    24
protocol                                        
P             0  14228  16697  13367     0  1591
S         18017  45757  29567  48183  2718  2914
X         20321  10126   2427  15863     0     0

Total cells with (protocol, age): 241776 / 241776


### Finding

**37/37 samples parse. 241,776 / 241,776 cells mapped.**

**Cell-level grid:**

| protocol \ age_week | 3 | 5 | 8 | 10 | 15 | 24 |
|---|---|---|---|---|---|---|
| P | 0 | 14,228 | 16,697 | 13,367 | 0 | 1,591 |
| S | 18,017 | 45,757 | 29,567 | 48,183 | 2,718 | 2,914 |
| X | 20,321 | 10,126 | 2,427 | 15,863 | 0 | 0 |

**Stratum health:** 14 populated strata out of 18 possible (4 empty: P×3, P×15, X×15, X×24). Lowest populated stratum = X×8 at **2,427 cells** — an order of magnitude above the 200-cell floor. **The hard-drop rule removes 0 cells from Bhaduri 2020.**

**Rough per-protocol share at 100k target (proportional):** S ≈ 63k, P ≈ 19k, X ≈ 19k. Reflects the true cohort composition — S was run in more cell lines across more timepoints.

## Section 5 — Bhaduri 2020: stratified subsample to 100k

With `protocol` and `age_week` derived in Section 4, we can run the subsample.

**Method:** proportional per-stratum targets via the largest-remainder rule (guarantees targets sum to exactly 100,000). Hard-drop any stratum with fewer than 200 cells — we already verified none of the 14 populated strata fall below this floor, but the rule is in the code for correctness (identical logic runs on Bhaduri 2021 in Section 6, where strata *do* drop). Fixed seed = 42.

First, write `protocol` and `age_week` back into `adata.obs` as proper columns (Section 4 built them in a temp DataFrame). Then compute per-stratum targets and display them before committing to the subsample — a visual check that numbers sum to 100,000 and match the expected S ≈ 63k, P ≈ 19k, X ≈ 19k split.

### 5a — Compute per-stratum targets

In [6]:
import numpy as np
import pandas as pd

SEED = 42
TARGET = 100_000
FLOOR = 200

# Write protocol/age_week into adata.obs as proper columns
adata.obs['protocol'] = sample_str.map(protocol_map).astype('category')
adata.obs['age_week'] = sample_str.map(age_map).astype('Int64')

# Stratum sizes (drop empty combinations)
strata = adata.obs.groupby(['protocol', 'age_week'], observed=True).size()
strata = strata[strata > 0]

# Apply hard-drop floor
passed = strata[strata >= FLOOR]
dropped = strata[strata < FLOOR]
print(f"Strata: {len(strata)} total, {len(passed)} passed (>={FLOOR}), {len(dropped)} dropped")
if len(dropped):
    print("Dropped:")
    print(dropped)
print(f"Cells available for sampling: {passed.sum()}")

# Largest-remainder proportional targets (guarantees sum = TARGET)
raw = passed * TARGET / passed.sum()
floor_t = np.floor(raw).astype(int)
remainder = TARGET - floor_t.sum()
frac = raw - floor_t
order = np.argsort(-frac.values)
targets = floor_t.copy()
targets.iloc[order[:remainder]] += 1

# Safety cap: target can't exceed stratum size
targets = pd.concat([targets.rename('target'), passed.rename('available')], axis=1)
targets['target'] = targets[['target', 'available']].min(axis=1)

print(f"\nPer-stratum targets (sum = {targets['target'].sum()}):")
print(targets)

# Per-protocol totals (quick sanity)
print("\nPer-protocol totals:")
print(targets.groupby(level='protocol')['target'].sum())

Strata: 14 total, 14 passed (>=200), 0 dropped
Cells available for sampling: 241776

Per-stratum targets (sum = 100000):
                   target  available
protocol age_week                   
P        5           5885      14228
         8           6906      16697
         10          5529      13367
         24           658       1591
S        3           7452      18017
         5          18925      45757
         8          12229      29567
         10         19929      48183
         15          1124       2718
         24          1205       2914
X        3           8405      20321
         5           4188      10126
         8           1004       2427
         10          6561      15863

Per-protocol totals:
protocol
P    18978
S    60864
X    20158
Name: target, dtype: int64


/tmp/ipykernel_3672/1817007079.py:43: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(targets.groupby(level='protocol')['target'].sum())


### Finding

All 14 populated strata passed the 200-cell floor — **0 dropped** (lowest stratum X×8 at 2,427 cells, ca. 10× above the floor). Per-stratum targets sum to exactly **100,000** (largest-remainder math).

**Per-protocol totals:** S = 60,864 (60.9%), X = 20,158 (20.2%), P = 18,978 (19.0%) — matches the cell-level grid from Section 4. The S protocol dominates because it was run in more cell lines across more timepoints.

Every target ≤ available, so sampling without replacement is safe. The uniform per-stratum rate is 100,000 / 241,776 = **41.4%** applied proportionally across all 14 strata (spot-checked: P×5 = 5,885/14,228, S×5 = 18,925/45,757, X×8 = 1,004/2,427 — all 41.4%).

### 5b — Execute subsample and save

One numpy RNG with fixed seed, one `rng.choice(..., replace=False)` call per passed stratum. Slicing `adata[keep_indices]` preserves `adata.raw` — raw counts are needed for downstream integration (colab_03 re-run on balanced input).

In [7]:
rng = np.random.default_rng(SEED)
keep_indices = []
for (p, a), row in targets.iterrows():
    mask = (adata.obs['protocol'] == p) & (adata.obs['age_week'] == a)
    stratum_idx = np.where(mask.values)[0]
    chosen = rng.choice(stratum_idx, size=int(row['target']), replace=False)
    keep_indices.extend(chosen.tolist())
keep_indices = sorted(keep_indices)

adata_sub = adata[keep_indices].copy()
print(f"Subsample shape: {adata_sub.shape}")
print(f"adata_sub.raw present: {adata_sub.raw is not None}")

# Composition sanity — proportions preserved?
print("\nProtocol composition (original vs subsample):")
print(pd.DataFrame({
    'original': adata.obs['protocol'].value_counts(normalize=True).sort_index().round(4),
    'subsample': adata_sub.obs['protocol'].value_counts(normalize=True).sort_index().round(4),
}))
print("\nAge_week composition (original vs subsample):")
print(pd.DataFrame({
    'original': adata.obs['age_week'].value_counts(normalize=True).sort_index().round(4),
    'subsample': adata_sub.obs['age_week'].value_counts(normalize=True).sort_index().round(4),
}))

# Save
OUT_PATH = '/content/drive/MyDrive/brain-organoid-trajectories/data/processed/bhaduri_2020/bhaduri_2020_100k.h5ad'
adata_sub.write_h5ad(OUT_PATH)
print(f"\nSaved: {OUT_PATH}")

# Free memory before Section 6
del adata, adata_sub
import gc; gc.collect()

Subsample shape: (100000, 16774)
adata_sub.raw present: True

Protocol composition (original vs subsample):
          original  subsample
protocol                     
P           0.1898     0.1898
S           0.6086     0.6086
X           0.2016     0.2016

Age_week composition (original vs subsample):
          original  subsample
age_week                     
3           0.1586     0.1586
5             0.29       0.29
8           0.2014     0.2014
10          0.3202     0.3202
15          0.0112     0.0112
24          0.0186     0.0186

Saved: /content/drive/MyDrive/brain-organoid-trajectories/data/processed/bhaduri_2020/bhaduri_2020_100k.h5ad


1149

### Finding

**Shape: (100,000, 16,774)** — exact target, all genes preserved.

`adata_sub.raw present: True` — raw counts survive the slice, so they're saved in the output h5ad. Integration (colab_03 re-run) will start from these raw counts.

**Protocol and age_week compositions match the original to 4 decimal places:**

| protocol | original | subsample |
|---|---|---|
| P | 0.1898 | 0.1898 |
| S | 0.6086 | 0.6086 |
| X | 0.2016 | 0.2016 |

Same for `age_week` (GW3 = 0.1586, GW5 = 0.29, GW8 = 0.2014, GW10 = 0.3202, GW15 = 0.0112, GW24 = 0.0186 — identical in both). Stratified proportional sampling preserved the joint distribution exactly.

File saved to `data/processed/bhaduri_2020/bhaduri_2020_100k.h5ad` (ca. 2.02 GB on disk, sparse format retained). `adata` and `adata_sub` are freed before Section 6 — Colab memory gets tight with 396k-cell loads next.

## Section 6 — Bhaduri 2021: load and apply pre-subsample filters

Load `bhaduri_2021_raw.h5ad` (396,186 × 33,694, built in colab_06 from 74 NeMO samples). Before stratifying we drop two cell populations flagged in the UCSC metadata:

- **`cell_type_coarse == "Outlier"`** — ca. 30k cells. UCSC's explicit quality-control flag; the atlas team excluded these from downstream analysis.
- **`cell_type == "0"`** — ca. 50k cells where the fine-grained consensus classifier could not assign an identity (literal string `"0"`).

Dropping these *before* stratification matters: if we included them, we'd either waste subsample slots on ambiguous cells or have to post-filter (which would break the exact 100,000 target).

After filtering, preview the `age_gw × cell_type_coarse` grid and apply the 200-cell floor. Unlike Bhaduri 2020 (14 populated strata, lowest at 2,427), this grid spans ca. 12 ages × 9 remaining coarse types = up to ca. 108 strata — so rare combinations (e.g. `GW14 × CR`, or late-stage vascular cells) will fall below the floor and drop.

### 6a — Load Bhaduri 2021 and inspect metadata

In [8]:
import scanpy as sc

adata21 = sc.read_h5ad('/content/drive/MyDrive/brain-organoid-trajectories/data/processed/bhaduri_2021/bhaduri_2021_raw.h5ad')
print("Shape:", adata21.shape)
print("\nobs columns:", list(adata21.obs.columns))
print("\nFirst 5 obs rows:")
print(adata21.obs.head())
print("\ncell_type_coarse distribution:")
print(adata21.obs['cell_type_coarse'].value_counts())
print("\ncell_type distribution (top 15):")
print(adata21.obs['cell_type'].value_counts().head(15))

Shape: (396186, 33694)

obs columns: ['source_tarball', 'ucsc_prefix', 'donor', 'area', 'age_gw', 'cell_type_coarse', 'Name', 'CombinedCluster - Iteration 1', 'cluster_final', 'area_ucsc', 'age_gw_ucsc', 'lamina', 'individual', 'cell_type', 'age_range', 'Main Brain Region']

First 5 obs rows:
                          source_tarball ucsc_prefix donor area  age_gw  \
GW14_V1_AAACCTGAGGGATGGG  GW14_occipital     GW14_V1  GW14   V1      14   
GW14_V1_AAACCTGTCATGTGGT  GW14_occipital     GW14_V1  GW14   V1      14   
GW14_V1_AAACCTGTCCGATATG  GW14_occipital     GW14_V1  GW14   V1      14   
GW14_V1_AAACCTGTCGTTACGA  GW14_occipital     GW14_V1  GW14   V1      14   
GW14_V1_AAACGGGCATGGTTGT  GW14_occipital     GW14_V1  GW14   V1      14   

                         cell_type_coarse                      Name  \
GW14_V1_AAACCTGAGGGATGGG           Neuron      GW14_Neuron_7_Neuron   
GW14_V1_AAACCTGTCATGTGGT         Dividing  GW14_Dividing_1_Dividing   
GW14_V1_AAACCTGTCCGATATG         Dividing 

### Finding

Shape **(396,186 × 33,694)** matches the colab_06 output. All required obs columns present: `cell_type_coarse`, `cell_type`, `age_gw`, plus UCSC merge columns (`area_ucsc`, `age_gw_ucsc`, `individual`).

**`cell_type_coarse` distribution matches Session 17 preflight exactly:** Neuron 50.8%, Interneuron 15.1%, RG 9.8%, Dividing 8.1%, Outlier 7.6%, IPC 3.4%, Microglia 2.5%, OPC 1.8%, Vascular 0.8%, CR 0.07%.

**`cell_type` quirks confirmed:**
- `"0"` placeholder: **50,447 cells** (UCSC's unassigned tag) — will be dropped.
- Only 12 unique values total — no subtype granularity beyond the coarse labels.
- `"Other"` (761) and `"Excitatory Neuron"` (61) are small residual categories that survive the filter.

`Main Brain Region` shows NaN in the head (sparsely populated, not used for stratification — safe to ignore).

### 6b — Drop Outlier and cell_type == "0"

Before stratifying, drop the two cell populations flagged in the UCSC metadata: `cell_type_coarse == "Outlier"` (UCSC's explicit QC flag — ca. 30k cells) and `cell_type == "0"` (ca. 50k cells where the fine-grained consensus classifier couldn't assign an identity). Dropping these *before* stratification matters: if we kept them, we'd either waste subsample slots on ambiguous cells or have to post-filter (which would break the exact 100,000 target).

In [9]:
n0 = adata21.n_obs
mask = (adata21.obs['cell_type_coarse'] != 'Outlier') & (adata21.obs['cell_type'] != '0')
adata21 = adata21[mask].copy()

dropped = n0 - adata21.n_obs
print(f"Dropped: {dropped:,} cells ({100 * dropped / n0:.1f}%)")
print(f"After filter: {adata21.shape}")

Dropped: 50,447 cells (12.7%)
After filter: (345739, 33694)


### 6c — Verify the filter worked

Quick sanity check before stratifying — confirm `Outlier` is no longer in the `cell_type_coarse` distribution, and see the per-cell-type counts we'll stratify across.

In [11]:
adata21.obs['cell_type_coarse'].value_counts()

,count
cell_type_coarse,
Neuron,180838
Interneuron,59955
RG,38835
Dividing,32033
IPC,13605
Microglia,9765
OPC,7293
Vascular,3149
CR,266


### Finding

`Outlier` is absent (0 cells) — confirms the filter worked. But the arithmetic reveals something the Session 17 notes got wrong: **Outlier ⊂ `cell_type == "0"`**, not disjoint. The 50,447 dropped cells decompose as:

- **30,117** Outlier (coarse) cells — all had `cell_type == "0"`. These are UCSC's QC-flagged cells (doublets / low-quality / ambiguous). The coarse "Outlier" label and the fine `"0"` placeholder co-label the same cells: if you don't trust a cell, you don't fine-annotate it.
- **20,330** Neuron (coarse) cells with `cell_type == "0"` — cells that passed QC as real neurons at the coarse level, but the fine classifier (12 cell types) couldn't confidently assign a subtype. Neuron dropped from 201,168 → 180,838 (ca. 10% of Neurons).

Net: **345,739 cells × 33,694 genes** retained, 9 coarse cell types (no Outlier). The original estimate of ca. 80,000 drops (30k + 50k as if disjoint) was wrong by the overlap; the real drop is just the union, 50,447.

### 6d — Build the grid and apply the 200-cell floor

With the filter applied and verified, build the cell-level `age_gw × cell_type_coarse` grid on the 345,739 filtered cells and apply the 200-cell floor. Bhaduri 2021 spans 7 timepoints × 9 coarse cell types = up to 63 possible strata — we expect some rare combinations (e.g. `GW14 × CR`, `GW14 × Microglia`) to fall below the floor and drop.

In [12]:
import pandas as pd
FLOOR = 200

# Cell-level grid: age_gw x cell_type_coarse
grid = adata21.obs.groupby(['age_gw', 'cell_type_coarse'], observed=True).size().unstack(fill_value=0)
print("Cell-level grid (age_gw x cell_type_coarse):")
print(grid)

# Floor check
strata21 = adata21.obs.groupby(['age_gw', 'cell_type_coarse'], observed=True).size()
strata21 = strata21[strata21 > 0]
passed21 = strata21[strata21 >= FLOOR]
dropped21 = strata21[strata21 < FLOOR]
print(f"\nStrata: {len(strata21)} total, {len(passed21)} passed (>={FLOOR}), {len(dropped21)} dropped")
if len(dropped21):
    print(f"\nDropped strata (totaling {dropped21.sum()} cells):")
    print(dropped21.sort_values())
print(f"\nCells in kept strata: {passed21.sum():,} / {strata21.sum():,} ({100 * passed21.sum() / strata21.sum():.1f}%)")

Cell-level grid (age_gw x cell_type_coarse):
cell_type_coarse   CR  Dividing   IPC  Interneuron  Microglia  Neuron   OPC  \
age_gw                                                                        
14                  0      1706    50            0         18    4906     0   
17                124       755   212            0          0    2141     0   
18                 46      9032  9004        17589        974   57326   639   
19                 96      5724  3758        14867       1050   20266   955   
20                  0      7790   581        19107       4932   40524   956   
22                  0      3834     0         4090        672   12789  1857   
25                  0      3192     0         4302       2119   42886  2886   

cell_type_coarse    RG  Vascular  
age_gw                            
14                1295        53  
17                 461       240  
18                9173       502  
19                6026       748  
20                6059       456 

### Finding

Grid: 7 age_gw × 9 cell types = 63 possible cells, **52 populated** (11 empty — rare combinations that didn't occur in the source data). Of the 52 populated strata, **6 fall below the 200-cell floor and drop, totaling 387 cells (0.11% of the filtered pool)**.

The dropped strata make developmental sense:
- **CR (Cajal-Retzius)** at GW17 (124), GW18 (46), GW19 (96) — CR is the rarest type overall (266 total), transiently produced early and then fades. Three of the six drops.
- **Microglia at GW14 (18)** — microglia colonize the fetal brain progressively; by GW18 they're at 974. 18 cells at GW14 is real biology, not undersampling.
- **IPC (50) and Vascular (53) at GW14** — early-development sparsity.

**Kept: 345,352 / 345,739 cells (99.9%)** for the 100k draw. All core populations (Neuron, Interneuron, RG, Dividing) are well-represented across GW18–25 with per-stratum counts in the thousands, so the subsample will hit them cleanly.

Dropping these fringe strata is the right trade: they're exactly the disconnected / isolated-node cases that broke DPT in Session 15. The 200-floor rule is doing its job.

## Section 7 — Bhaduri 2021: stratified subsample to 100k

Mirror of Section 5 applied to the filtered Bhaduri 2021 (`passed21` from Section 6). Same method: largest-remainder proportional targets, hard-drop floor already applied in Section 6, fixed seed = 42.

The sampling is independent from Bhaduri 2020, but the seed is the same — this just means both draws are deterministic, not that they're correlated (they operate on disjoint data).

### 7a — Compute per-stratum targets

In [13]:
import numpy as np
import pandas as pd

SEED = 42
TARGET = 100_000

# passed21 defined in Section 6; strata already floor-filtered
raw = passed21 * TARGET / passed21.sum()
floor_t = np.floor(raw).astype(int)
remainder = TARGET - floor_t.sum()
frac = raw - floor_t
order = np.argsort(-frac.values)
targets21 = floor_t.copy()
targets21.iloc[order[:remainder]] += 1

# Safety cap: target can't exceed stratum size
targets21 = pd.concat([targets21.rename('target'), passed21.rename('available')], axis=1)
targets21['target'] = targets21[['target', 'available']].min(axis=1)

print(f"Per-stratum targets (sum = {targets21['target'].sum()}):")
print(targets21)
print(f"\nPer-age_gw totals:")
print(targets21.groupby(level='age_gw')['target'].sum())
print(f"\nPer-cell_type_coarse totals:")
print(targets21.groupby(level='cell_type_coarse')['target'].sum())

Per-stratum targets (sum = 100000):
                         target  available
age_gw cell_type_coarse                   
14     Dividing             494       1706
       Neuron              1421       4906
       RG                   375       1295
17     Dividing             219        755
       IPC                   61        212
       Neuron               620       2141
       RG                   133        461
       Vascular              70        240
18     Dividing            2615       9032
       IPC                 2607       9004
       Interneuron         5093      17589
       Microglia            282        974
       Neuron             16599      57326
       OPC                  185        639
       RG                  2656       9173
       Vascular             145        502
19     Dividing            1657       5724
       IPC                 1088       3758
       Interneuron         4305      14867
       Microglia            304       1050
       Neuron     

/tmp/ipykernel_3672/2743232585.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(targets21.groupby(level='cell_type_coarse')['target'].sum())


### Finding

Per-stratum targets sum to exactly **100,000** (largest-remainder). All 46 kept strata have target ≤ available, so sampling without replacement is safe. Uniform per-stratum rate is 100,000 / 345,352 = **28.96%** applied proportionally across every stratum.

**One meaningful loss to flag: CR = 0 in the targets.** All three CR strata (GW17 = 124, GW18 = 46, GW19 = 96) fell below the 200 floor in Section 6c, so the 100k subsample contains **zero Cajal-Retzius cells**. Bhaduri 2021 had only 266 CR total — too sparse to survive stratified sampling at this scale. Downstream: CR won't appear in integrated / annotated space. This is unavoidable given the hard-drop rule (necessary to protect DPT from zero-distance duplicates), but worth knowing when interpreting trajectory results.

**Per-age_gw targets (sum = 100k):** GW18 = 30,182 (30.2%) dominates, GW17 = 1,103 (1.1%) tiny, GW14 = 2,290 (2.3%) small, GW19 = 15,461, GW20 = 23,282, GW22 = 8,614, GW25 = 19,068. These match the source biology — early timepoints were genuinely undersampled in Bhaduri 2021, not a stratification artifact.

**Per-cell-type composition (100k):** Neuron 52,363 (52.4%), Interneuron 17,361 (17.4%), RG 11,244 (11.2%), Dividing 9,275 (9.3%), IPC 3,924 (3.9%), Microglia 2,823 (2.8%), OPC 2,113 (2.1%), Vascular 897 (0.9%), CR 0.

### 7b — Execute subsample and save

Same pattern as Section 5: one numpy RNG seeded at 42, one `rng.choice(..., replace=False)` per passed stratum. Save to `bhaduri_2021_100k.h5ad`, then free memory before Section 8.

In [14]:
rng = np.random.default_rng(SEED)
keep21 = []
for (gw, ct), row in targets21.iterrows():
    mask = (adata21.obs['age_gw'] == gw) & (adata21.obs['cell_type_coarse'] == ct)
    stratum_idx = np.where(mask.values)[0]
    chosen = rng.choice(stratum_idx, size=int(row['target']), replace=False)
    keep21.extend(chosen.tolist())
keep21 = sorted(keep21)

adata21_sub = adata21[keep21].copy()
print(f"Subsample shape: {adata21_sub.shape}")
print(f"adata21_sub.raw present: {adata21_sub.raw is not None}")

# Composition sanity (filtered original vs subsample)
print("\nage_gw composition (filtered original vs subsample):")
print(pd.DataFrame({
    'original': adata21.obs['age_gw'].value_counts(normalize=True).sort_index().round(4),
    'subsample': adata21_sub.obs['age_gw'].value_counts(normalize=True).sort_index().round(4),
}))
print("\ncell_type_coarse composition (filtered original vs subsample):")
print(pd.DataFrame({
    'original': adata21.obs['cell_type_coarse'].value_counts(normalize=True).sort_index().round(4),
    'subsample': adata21_sub.obs['cell_type_coarse'].value_counts(normalize=True).sort_index().round(4),
}))

# Save
OUT21 = '/content/drive/MyDrive/brain-organoid-trajectories/data/processed/bhaduri_2021/bhaduri_2021_100k.h5ad'
adata21_sub.write_h5ad(OUT21)
print(f"\nSaved: {OUT21}")

# Free memory before Section 8
del adata21, adata21_sub
import gc; gc.collect()

Subsample shape: (100000, 33694)
adata21_sub.raw present: False

age_gw composition (filtered original vs subsample):
        original  subsample
age_gw                     
14        0.0232     0.0229
17        0.0114     0.0110
18        0.3016     0.3018
19        0.1547     0.1546
20        0.2326     0.2328
22        0.0860     0.0861
25        0.1905     0.1907

cell_type_coarse composition (filtered original vs subsample):
                  original  subsample
cell_type_coarse                     
CR                  0.0008        NaN
Dividing            0.0927     0.0928
IPC                 0.0394     0.0392
Interneuron         0.1734     0.1736
Microglia           0.0282     0.0282
Neuron              0.5230     0.5236
OPC                 0.0211     0.0211
RG                  0.1123     0.1124
Vascular            0.0091     0.0090

Saved: /content/drive/MyDrive/brain-organoid-trajectories/data/processed/bhaduri_2021/bhaduri_2021_100k.h5ad


12890

### Finding

**Shape: (100,000, 33,694)** — exact target.

`adata21_sub.raw present: False` — different from Bhaduri 2020 (`raw = True`), but **expected**: Bhaduri 2021 came from `bhaduri_2021_raw.h5ad` which was never preprocessed, so `.X` itself holds the raw counts (no separate `.raw` slot needed). For colab_03: do NOT call `adata21.raw.to_adata()` — use `adata21` directly.

**Composition preserved to 4 decimals** on both axes:
- `age_gw`: GW18 0.3016 → 0.3018, GW17 0.0114 → 0.0110, GW14 0.0232 → 0.0229, etc.
- `cell_type_coarse`: matches exactly, with **`CR: NaN` in the subsample** — confirms the 0-count drop predicted in 7a.

File saved to `data/processed/bhaduri_2021/bhaduri_2021_100k.h5ad` (ca. 1.64 GB on disk, sparse format retained — *smaller* than Bhaduri 2020's 2.02 GB despite 2× the genes, reflecting Bhaduri 2021's lower median n_genes/cell of 1,228).

## Section 8 — Joint sanity checks

Load both 100k files back from Drive and verify:

- **Shape parity** — both should be `(100_000, N_genes)`.
- **Gene-space overlap** — the intersection is what `colab_03` will use for integration. Bhaduri 2020 has 16,774 genes; Bhaduri 2021 has 33,694. Expect the intersection to be close to the smaller set.
- **obs column inventory** — confirm stratification axes survived the save round-trip. Columns don't need to match between datasets; `colab_03` will add a `dataset` label during concatenation.

### 8a — Joint sanity checks on both 100k files

In [15]:
import scanpy as sc

adata20 = sc.read_h5ad('/content/drive/MyDrive/brain-organoid-trajectories/data/processed/bhaduri_2020/bhaduri_2020_100k.h5ad')
adata21 = sc.read_h5ad('/content/drive/MyDrive/brain-organoid-trajectories/data/processed/bhaduri_2021/bhaduri_2021_100k.h5ad')

print("Bhaduri 2020 (organoids):", adata20.shape)
print("  obs columns:", list(adata20.obs.columns))
print("  raw present:", adata20.raw is not None)
print()
print("Bhaduri 2021 (fetal):", adata21.shape)
print("  obs columns:", list(adata21.obs.columns))
print("  raw present:", adata21.raw is not None)

# Gene-space overlap
genes20 = set(adata20.var_names)
genes21 = set(adata21.var_names)
shared = genes20 & genes21
print(f"\nGene space:")
print(f"  Bhaduri 2020: {len(genes20):,} genes")
print(f"  Bhaduri 2021: {len(genes21):,} genes")
print(f"  Shared: {len(shared):,}")
print(f"  Bhaduri 2020 only: {len(genes20 - genes21):,}")
print(f"  Bhaduri 2021 only: {len(genes21 - genes20):,}")

Bhaduri 2020 (organoids): (100000, 16774)
  obs columns: ['n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'sample', 'leiden', 'protocol', 'age_week']
  raw present: True

Bhaduri 2021 (fetal): (100000, 33694)
  obs columns: ['source_tarball', 'ucsc_prefix', 'donor', 'area', 'age_gw', 'cell_type_coarse', 'Name', 'CombinedCluster - Iteration 1', 'cluster_final', 'area_ucsc', 'age_gw_ucsc', 'lamina', 'individual', 'cell_type', 'age_range', 'Main Brain Region']
  raw present: False

Gene space:
  Bhaduri 2020: 16,774 genes
  Bhaduri 2021: 33,694 genes
  Shared: 16,768
  Bhaduri 2020 only: 6
  Bhaduri 2021 only: 16,926


### 8b — What are the 6 Bhaduri-2020-only genes?

Only 6 out of 16,774 genes in Bhaduri 2020 aren't in Bhaduri 2021 — a trivial miss rate. Worth a quick spot-check to make sure none are biologically important markers (which would compromise the integration) rather than annotation artifacts.

In [16]:
only_20 = sorted(set(adata20.var_names) - set(adata21.var_names))
print("Bhaduri 2020-only genes (6):")
for g in only_20:
    print(f"  {g}")

Bhaduri 2020-only genes (6):
  CCDC7.1
  CYB561D2.1
  LINC01481.1
  MATR3.1
  PGM5-AS1.1
  RGS5.1


### Finding

Both files load cleanly at **(100,000, N_genes)**.

**Raw-slot asymmetry (expected):**
- Bhaduri 2020: `raw = True`, `.X` = normalized, `.raw.X` = counts (from preprocessed clustered file)
- Bhaduri 2021: `raw = False`, `.X` = counts directly (from untreated raw file)
- Action for colab_03: for 2020 call `adata20.raw.to_adata()` to get counts; for 2021 use `adata21` directly.

**Gene-space overlap — favorable:**
- Shared: **16,768 genes** (the integration intersection — what colab_03 uses)
- Bhaduri 2020 only: **6 genes** (99.96% of 2020's gene space is in 2021)
- Bhaduri 2021 only: 16,926 genes (half of 2021's gene space lost at intersection)

The asymmetry reflects different reference annotations — Bhaduri 2021 was likely reprocessed with a fuller GENCODE annotation (33,694 genes is typical of newer 10x pipelines including non-coding biotypes). Not a quality issue, just annotation scope.

**Vs the old Zhong integration:** colab_03 had 14,498 shared genes (Bhaduri 2020 × Zhong 2018). Now **16,768 shared — ca. 15% more** genes to work with for HVG selection and integration.

**The 6 Bhaduri-2020-only genes** (`CCDC7.1`, `CYB561D2.1`, `LINC01481.1`, `MATR3.1`, `PGM5-AS1.1`, `RGS5.1`) — all carry the `.1` suffix, a Cell Ranger artifact: when two Ensembl IDs in the reference GTF map to the same gene symbol, `cellranger mkref` appends `.1`, `.2`, etc. to disambiguate. The unsuffixed versions (`RGS5`, `MATR3`, etc.) exist in both datasets; these `.1` entries are just duplicate-row copies that Bhaduri 2021's reference resolved differently. **No biologically meaningful loss** — none are canonical neural markers (no SOX2, EOMES, TBR1, NEUROD2, GAD1/2, GFAP, MKI67, SATB2, HOPX, PAX6, etc.). The 16,768-gene intersection is clean for integration.

## Notebook complete

Deliverables:
- `data/processed/bhaduri_2020/bhaduri_2020_100k.h5ad`
- `data/processed/bhaduri_2021/bhaduri_2021_100k.h5ad`

**Next notebook:** re-run `colab_03_integration.ipynb` on these balanced inputs — concatenate on shared gene space, add `dataset` label, normalize, HVG, PCA, Harmony, Leiden, save `integrated_harmony.h5ad`. Then re-run `colab_04` (annotation) and `colab_05` (trajectory) on the balanced integration. The 100×-imbalance failure mode that broke DPT in Session 15 should be gone.